In [3]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx")

df["Plan"] = pd.to_numeric(df["Plan"], errors="coerce").fillna(0)
df["Cycle Time"] = pd.to_numeric(df["Cycle Time"], errors="coerce").fillna(0)

# =============================
# Constants
# =============================
USABLE_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# Tracking structures
# =============================
machine_load = {m: 0 for m in USABLE_MACHINES}
machine_plan = defaultdict(list)
rejection_log = []

# =============================
# Allocation logic
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    required_qty = row["Plan"]
    cycle_time = row["Cycle Time"]

    if required_qty <= 0 or cycle_time <= 0:
        continue

    required_time = required_qty * cycle_time

    vertical_machines = str(row["Vertical Mchines"]).split(",")
    vertical_machines = [m.strip() for m in vertical_machines]

    eligible = [m for m in vertical_machines if m in USABLE_MACHINES]

    if not eligible:
        rejection_log.append((child, "No eligible 120T machine"))
        continue

    remaining_time = required_time

    eligible.sort(key=lambda m: machine_load[m])

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejection_log.append((
            child,
            f"Shortfall qty {round(remaining_time / cycle_time, 2)}"
        ))

# =============================
# DISPLAY RESULTS
# =============================

print("\n================ MACHINE-WISE PLAN ================\n")
for m, plans in machine_plan.items():
    print(f"🔧 {m}")
    display(pd.DataFrame(plans))
    print("-" * 60)

print("\n================ MACHINE LOAD SUMMARY ================\n")
load_df = pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in USABLE_MACHINES
])
display(load_df)

print("\n================ REJECTIONS / SHORTFALLS ================\n")
if rejection_log:
    display(pd.DataFrame(rejection_log, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections. All plans feasible.")


================ MACHINE-WISE PLAN ================


================ MACHINE LOAD SUMMARY ================



,Machine,Used (min),Remaining (min)
0,MP-01,0,3960
1,MP-05,0,3960
2,MP-10,0,3960
3,MP-17,0,3960



================ REJECTIONS / SHORTFALLS ================



,Child Part,Reason
0,14CL510026-00001X1,No eligible 120T machine
1,14CL510026-00002X0,No eligible 120T machine
2,14CL510026-00004X0,No eligible 120T machine
3,14CL510026-00005X0,No eligible 120T machine
4,14CL510026-00007X0,No eligible 120T machine
...,...,...
2086,W01006-001A1X,No eligible 120T machine
2087,W01007-001A1X,No eligible 120T machine
2088,W01007-001A1X,No eligible 120T machine
2089,W01007-001A1X,No eligible 120T machine


In [4]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx")

df["Plan"] = pd.to_numeric(df["Plan"], errors="coerce").fillna(0)
df["Cycle Time"] = pd.to_numeric(df["Cycle Time"], errors="coerce").fillna(0)

# =============================
# EXPLICIT MACHINE ALLOW LIST
# =============================
ALLOWED_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# Tracking
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejection_log = []

# =============================
# Allocation
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    required_qty = row["Plan"]
    cycle_time = row["Cycle Time"]

    if required_qty <= 0 or cycle_time <= 0:
        continue

    required_time = required_qty * cycle_time

    vertical_machines = [
        m.strip() for m in str(row["Vertical Mchines"]).split(",")
    ]

    # STRICT FILTER — name-based only
    eligible = [m for m in vertical_machines if m in ALLOWED_MACHINES]

    if not eligible:
        rejection_log.append((child, "No allowed machine"))
        continue

    remaining_time = required_time

    # Load balancing
    eligible.sort(key=lambda m: machine_load[m])

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejection_log.append((
            child,
            f"Shortfall qty {round(remaining_time / cycle_time, 2)}"
        ))

# =============================
# DISPLAY RESULTS
# =============================

print("\n========== MACHINE-WISE PLAN ==========\n")
for m in sorted(ALLOWED_MACHINES):
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No assigned parts")
    print("-" * 50)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in sorted(ALLOWED_MACHINES)
]))

print("\n========== REJECTIONS ==========\n")
if rejection_log:
    display(pd.DataFrame(rejection_log, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")


========== MACHINE-WISE PLAN ==========

🔧 MP-01
No assigned parts
--------------------------------------------------
🔧 MP-05
No assigned parts
--------------------------------------------------
🔧 MP-10
No assigned parts
--------------------------------------------------
🔧 MP-17
No assigned parts
--------------------------------------------------

========== MACHINE LOAD SUMMARY ==========



,Machine,Used (min),Remaining (min)
0,MP-01,0,3960
1,MP-05,0,3960
2,MP-10,0,3960
3,MP-17,0,3960



========== REJECTIONS ==========



,Child Part,Reason
0,14CL510026-00001X1,No allowed machine
1,14CL510026-00002X0,No allowed machine
2,14CL510026-00004X0,No allowed machine
3,14CL510026-00005X0,No allowed machine
4,14CL510026-00007X0,No allowed machine
...,...,...
2086,W01006-001A1X,No allowed machine
2087,W01007-001A1X,No allowed machine
2088,W01007-001A1X,No allowed machine
2089,W01007-001A1X,No allowed machine


In [6]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE
# ────────────────────────────────────────────────
FILE_PATH = r"C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"   # ← your file path
SHEET_NAME = "Master Data "                                             # ← change if needed
OUTPUT_FILE = "production_plan_120ton.xlsx"                       # where to save result

# Machine group (excluding fixed-tool machines MP-11, MP-15)
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Productive seconds per day per machine (8 hours = 8*60*60)
SECONDS_PER_DAY = 8 * 3600
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS

# Column names in your file (change only if they are different)
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_DAILY_PLAN  = "Daily Plan"
COL_MONTHLY_REQ = "Monthly Requirement"
COL_MIN_QTY     = "Minimum Quantity"
COL_PLAN_QTY    = "Plan"
COL_CYCLE_TIME  = "Cycle Time"
COL_MACHINE     = "Vertical Machines"
# ────────────────────────────────────────────────

def load_data():
    if FILE_PATH.lower().endswith('.csv'):
        df = pd.read_csv(FILE_PATH)
    else:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    
    # Clean column names (remove extra spaces)
    df.columns = df.columns.str.strip()
    
    # Keep only relevant columns
    keep_cols = [COL_CHILD, COL_SWITCH, COL_DAILY_PLAN, COL_MONTHLY_REQ,
                 COL_MIN_QTY, COL_PLAN_QTY, COL_CYCLE_TIME, COL_MACHINE]
    df = df[keep_cols].copy()
    
    # Convert numeric columns
    for col in [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MONTHLY_REQ, COL_MIN_QTY, COL_DAILY_PLAN]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.dropna(subset=[COL_PLAN_QTY, COL_CYCLE_TIME])  # must have qty & time
    
    return df


def calculate_loads(df):
    # Calculate production seconds needed
    df['Production_Seconds'] = df[COL_PLAN_QTY] * df[COL_CYCLE_TIME]
    df['Production_Hours']   = df['Production_Seconds'] / 3600
    
    # Normalize machine names
    df[COL_MACHINE] = df[COL_MACHINE].str.strip().str.upper()
    
    # Split into fixed and flexible
    fixed = df[df[COL_MACHINE].isin(ALLOWED_MACHINES)].copy()
    flexible = df[~df[COL_MACHINE].isin(ALLOWED_MACHINES) & 
                  (df[COL_MACHINE].str.lower() != 'fixed') & 
                  df[COL_MACHINE].notna()].copy()  # assume others are flexible
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    # Greedy load balancing: assign to machine with current lowest load
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}        # in seconds
    assignments = defaultdict(list)
    
    # Sort by biggest jobs first (helps balance better)
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        # Find machine with lowest current load
        best_machine = min(machine_load, key=machine_load.get)
        assignments[best_machine].append(row)
        machine_load[best_machine] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    # Fixed assignments
    for _, row in fixed.iterrows():
        m = row[COL_MACHINE]
        plan_rows.append({
            'Machine': m,
            'Child Part': row[COL_CHILD],
            'Switch Part': row[COL_SWITCH],
            'Plan Qty': row[COL_PLAN_QTY],
            'Cycle Time (s)': row[COL_CYCLE_TIME],
            'Total Seconds': row['Production_Seconds'],
            'Total Hours': row['Production_Hours'],
            'Type': 'Fixed'
        })
    
    # Flexible assignments
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Part': part[COL_SWITCH],
                'Plan Qty': part[COL_PLAN_QTY],
                'Cycle Time (s)': part[COL_CYCLE_TIME],
                'Total Seconds': part['Production_Seconds'],
                'Total Hours': part['Production_Hours'],
                'Type': 'Flexible'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Summary per machine
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Utilization %'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Available Hours'] = AVAILABLE_SECONDS_PER_MACHINE / 3600
    summary = summary[['Part Count', 'Total Hours', 'Available Hours', 'Utilization %']]
    
    return plan_df, summary


def main():
    print("Loading data...")
    df = load_data()
    
    print(f"Total Child Parts found: {len(df)}")
    
    fixed, flexible = calculate_loads(df)
    print(f"Fixed parts  : {len(fixed)}")
    print(f"Flexible parts: {len(flexible)}")
    
    flexible_assignments, machine_load_sec = assign_flexible_parts(flexible)
    
    plan_df, summary = create_final_plan(fixed, flexible_assignments)
    
    # Save to Excel
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_df.to_excel(writer, sheet_name='Detailed Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
    
    print("\n" + "="*60)
    print("           PRODUCTION PLAN CREATED")
    print("="*60)
    print("\nMachine Summary (3-day plan):")
    print(summary.round(2))
    
    print(f"\nDetailed plan saved to: {OUTPUT_FILE}")
    print("\nColumns in detailed sheet:")
    print(plan_df.columns.tolist())
    
    # Quick check for overload
    overloaded = summary[summary['Utilization %'] > 100]
    if not overloaded.empty:
        print("\nWARNING: Overloaded machines!")
        print(overloaded)


if __name__ == "__main__":
    main()

Loading data...
Total Child Parts found: 2232
Fixed parts  : 0
Flexible parts: 436

           PRODUCTION PLAN CREATED

Machine Summary (3-day plan):
         Part Count  Total Hours  Available Hours  Utilization %
Machine                                                         
MP-01           107      1522.56             24.0         6344.0
MP-05           115      1499.35             24.0         6247.3
MP-10           107      1522.56             24.0         6344.0
MP-17           107      1522.56             24.0         6344.0

Detailed plan saved to: production_plan_120ton.xlsx

Columns in detailed sheet:
['Machine', 'Child Part', 'Switch Part', 'Plan Qty', 'Cycle Time (s)', 'Total Seconds', 'Total Hours', 'Type']

         Part Count  Total Hours  Available Hours  Utilization %
Machine                                                         
MP-01           107  1522.559677             24.0         6344.0
MP-05           115  1499.354834             24.0         6247.3
MP-10  

In [7]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE
# ────────────────────────────────────────────────
FILE_PATH = r"C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"   # ← CHANGE THIS to your actual file path
SHEET_NAME = "Master Data "                                             # ← change if needed
OUTPUT_FILE = "production_plan_120ton.xlsx"                      # output file name

# Machines in 120 tonnage group (excluding MP-11 and MP-15)
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Productive seconds per day per machine (8 hours/day)
SECONDS_PER_DAY = 8 * 3600
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS

# Column names in your Excel/CSV file (adjust only if names are different)
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_DAILY_PLAN  = "Daily Plan"
COL_MONTHLY_REQ = "Monthly Requirement"
COL_MIN_QTY     = "Minimum Quantity"
COL_PLAN_QTY    = "Plan"
COL_CYCLE_TIME  = "Cycle Time"
COL_MACHINE     = "Vertical Machines"
# ────────────────────────────────────────────────

def load_and_aggregate_data():
    # Read file
    if FILE_PATH.lower().endswith('.csv'):
        df = pd.read_csv(FILE_PATH)
    else:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    
    # Clean column names (remove extra spaces)
    df.columns = df.columns.str.strip()
    
    # Keep only needed columns
    keep_cols = [COL_CHILD, COL_SWITCH, COL_DAILY_PLAN, COL_MONTHLY_REQ,
                 COL_MIN_QTY, COL_PLAN_QTY, COL_CYCLE_TIME, COL_MACHINE]
    df = df[keep_cols].copy()
    
    # Convert numeric columns
    numeric_cols = [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MONTHLY_REQ, COL_MIN_QTY, COL_DAILY_PLAN]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop rows missing critical data
    df = df.dropna(subset=[COL_CHILD, COL_PLAN_QTY, COL_CYCLE_TIME])
    
    # ──── IMPORTANT: Take FIRST occurrence of each Child Part ─────
    # Preserve original file order and take first row per unique Child Part
    df = df.sort_index()  # ensure original order
    aggregated = df.groupby(COL_CHILD, as_index=False).first()
    
    # Add count of how many times each part appeared (for your info)
    counts = df[COL_CHILD].value_counts().reset_index(name='Appearance_Count')
    aggregated = aggregated.merge(counts, on=COL_CHILD, how='left')
    
    # Rename columns for clarity
    aggregated = aggregated.rename(columns={
        COL_PLAN_QTY:    'Plan_Qty',
        COL_CYCLE_TIME:  'Cycle_Time_s',
        COL_MACHINE:     'Assigned_Machine',
        COL_SWITCH:      'Switch_Part_Reference',
        COL_MONTHLY_REQ: 'Monthly_Req',
        COL_MIN_QTY:     'Min_Qty'
    })
    
    print(f"Original rows: {len(df)}")
    print(f"Unique Child Parts (using first occurrence): {len(aggregated)}")
    print(f"Parts that appeared multiple times: {len(aggregated[aggregated['Appearance_Count'] > 1])}")
    
    return aggregated


def calculate_loads(agg_df):
    agg_df['Production_Seconds'] = agg_df['Plan_Qty'] * agg_df['Cycle_Time_s']
    agg_df['Production_Hours']   = agg_df['Production_Seconds'] / 3600
    
    # Normalize machine names
    agg_df['Assigned_Machine'] = agg_df['Assigned_Machine'].astype(str).str.strip().str.upper()
    
    # Split into fixed and flexible
    fixed_mask = agg_df['Assigned_Machine'].isin(ALLOWED_MACHINES)
    fixed = agg_df[fixed_mask].copy()
    flexible = agg_df[~fixed_mask].copy()
    
    print(f"Fixed parts   : {len(fixed)}")
    print(f"Flexible parts: {len(flexible)}")
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    # Greedy balancing: assign biggest jobs first to least loaded machine
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}  # current load in seconds
    assignments = defaultdict(list)
    
    # Sort descending by production time (helps better balance)
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        best_machine = min(machine_load, key=machine_load.get)
        assignments[best_machine].append(row)
        machine_load[best_machine] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    # Fixed parts
    for _, row in fixed.iterrows():
        plan_rows.append({
            'Machine': row['Assigned_Machine'],
            'Child Part': row[COL_CHILD],
            'Switch Reference': row['Switch_Part_Reference'],
            'Plan Qty': row['Plan_Qty'],
            'Cycle Time (s)': row['Cycle_Time_s'],
            'Total Seconds': row['Production_Seconds'],
            'Total Hours': round(row['Production_Hours'], 2),
            'Type': 'Fixed'
        })
    
    # Flexible assignments
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Reference': part['Switch_Part_Reference'],
                'Plan Qty': part['Plan_Qty'],
                'Cycle Time (s)': part['Cycle_Time_s'],
                'Total Seconds': part['Production_Seconds'],
                'Total Hours': round(part['Production_Hours'], 2),
                'Type': 'Flexible'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Machine summary
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Utilization %'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Available Hours (3 days)'] = AVAILABLE_SECONDS_PER_MACHINE / 3600
    
    summary = summary[['Part Count', 'Total Hours', 'Available Hours (3 days)', 'Utilization %']]
    
    return plan_df, summary


def main():
    print("Processing production plan for 120-ton vertical machines...\n")
    
    agg_df = load_and_aggregate_data()
    fixed, flexible = calculate_loads(agg_df)
    
    flexible_assignments, machine_load_sec = assign_flexible_parts(flexible)
    
    plan_df, summary = create_final_plan(fixed, flexible_assignments)
    
    # Save to Excel
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_df.to_excel(writer, sheet_name='Detailed Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
    
    print("\n" + "="*70)
    print("         PRODUCTION PLAN GENERATED SUCCESSFULLY")
    print("="*70)
    
    print("\nMachine Summary (3-day horizon):")
    print(summary.round(2))
    
    print(f"\nDetailed plan saved to: {OUTPUT_FILE}")
    
    # Overload warning
    overloaded = summary[summary['Utilization %'] > 100]
    if not overloaded.empty:
        print("\nWARNING ─ OVERLOADED MACHINES:")
        print(overloaded)
    else:
        print("\nAll machines under 100% utilization → good balance.")


if __name__ == "__main__":
    main()

Processing production plan for 120-ton vertical machines...

Original rows: 2232
Unique Child Parts (using first occurrence): 560
Parts that appeared multiple times: 401
Fixed parts   : 0
Flexible parts: 560

         PRODUCTION PLAN GENERATED SUCCESSFULLY

Machine Summary (3-day horizon):
         Part Count  Total Hours  Available Hours (3 days)  Utilization %
Machine                                                                  
MP-01           127      4593.03                      24.0        19137.7
MP-05           177      4503.73                      24.0        18765.5
MP-10           128      4593.04                      24.0        19137.7
MP-17           128      4593.03                      24.0        19137.7

Detailed plan saved to: production_plan_120ton.xlsx

WARNING ─ OVERLOADED MACHINES:
         Part Count  Total Hours  Available Hours (3 days)  Utilization %
Machine                                                                  
MP-01           127      4593.03

In [8]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE AS NEEDED
# ────────────────────────────────────────────────
FILE_PATH = r"C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"   # ← CHANGE THIS to your actual file path
SHEET_NAME = "Master Data "                                             # ← change if needed
OUTPUT_FILE = "production_plan_120ton.xlsx"           # where results will be saved

# 120 tonnage vertical machines (excluding fixed-tool machines MP-11 & MP-15)
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Production capacity
HOURS_PER_DAY = 22
SECONDS_PER_DAY = HOURS_PER_DAY * 3600                   # 79,200 seconds/day
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS   # 237,600 seconds
AVAILABLE_HOURS_PER_MACHINE   = HOURS_PER_DAY * PLANNING_DAYS     # 66 hours

# Your column names (adjust only if they differ exactly)
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_DAILY_PLAN  = "Daily Plan"
COL_MONTHLY_REQ = "Monthly Requirement"
COL_MIN_QTY     = "Minimum Quantity"
COL_PLAN_QTY    = "Plan"
COL_CYCLE_TIME  = "Cycle Time"
COL_MACHINE     = "Vertical Machines"
# ────────────────────────────────────────────────

def load_and_aggregate_data():
    if FILE_PATH.lower().endswith('.csv'):
        df = pd.read_csv(FILE_PATH)
    else:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    
    df.columns = df.columns.str.strip()
    
    keep_cols = [COL_CHILD, COL_SWITCH, COL_DAILY_PLAN, COL_MONTHLY_REQ,
                 COL_MIN_QTY, COL_PLAN_QTY, COL_CYCLE_TIME, COL_MACHINE]
    df = df[keep_cols].copy()
    
    numeric_cols = [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MONTHLY_REQ, COL_MIN_QTY, COL_DAILY_PLAN]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    df = df.dropna(subset=[COL_CHILD, COL_PLAN_QTY, COL_CYCLE_TIME])
    
    # Take FIRST occurrence of each Child Part only (no summing)
    df = df.sort_index()  # preserve original file order
    aggregated = df.groupby(COL_CHILD, as_index=False).first()
    
    # Show duplication info
    counts = df[COL_CHILD].value_counts().reset_index(name='Appearance_Count')
    aggregated = aggregated.merge(counts, on=COL_CHILD, how='left')
    
    aggregated = aggregated.rename(columns={
        COL_PLAN_QTY:    'Plan_Qty',
        COL_CYCLE_TIME:  'Cycle_Time_s',
        COL_MACHINE:     'Assigned_Machine',
        COL_SWITCH:      'Switch_Reference',
        COL_MONTHLY_REQ: 'Monthly_Req',
        COL_MIN_QTY:     'Min_Qty'
    })
    
    print(f"Original rows in file     : {len(df)}")
    print(f"Unique Child Parts        : {len(aggregated)}")
    print(f"Parts duplicated (≥2x)    : {len(aggregated[aggregated['Appearance_Count'] > 1])}")
    
    return aggregated


def calculate_loads(agg_df):
    agg_df['Production_Seconds'] = agg_df['Plan_Qty'] * agg_df['Cycle_Time_s']
    agg_df['Production_Hours']   = agg_df['Production_Seconds'] / 3600
    
    agg_df['Assigned_Machine'] = agg_df['Assigned_Machine'].astype(str).str.strip().str.upper()
    
    fixed = agg_df[agg_df['Assigned_Machine'].isin(ALLOWED_MACHINES)].copy()
    flexible = agg_df[~agg_df['Assigned_Machine'].isin(ALLOWED_MACHINES)].copy()
    
    print(f"Fixed-assigned parts      : {len(fixed)}")
    print(f"Flexible parts to assign  : {len(flexible)}")
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}  # current load in seconds
    assignments = defaultdict(list)
    
    # Sort largest jobs first → better balance
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        best_machine = min(machine_load, key=machine_load.get)
        assignments[best_machine].append(row)
        machine_load[best_machine] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    # Fixed
    for _, row in fixed.iterrows():
        plan_rows.append({
            'Machine': row['Assigned_Machine'],
            'Child Part': row[COL_CHILD],
            'Switch Reference': row['Switch_Reference'],
            'Plan Qty (3 days)': row['Plan_Qty'],
            'Cycle Time (s)': row['Cycle_Time_s'],
            'Total Seconds': int(row['Production_Seconds']),
            'Total Hours': round(row['Production_Hours'], 2),
            'Assignment Type': 'Fixed'
        })
    
    # Flexible
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Reference': part['Switch_Reference'],
                'Plan Qty (3 days)': part['Plan_Qty'],
                'Cycle Time (s)': part['Cycle_Time_s'],
                'Total Seconds': int(part['Production_Seconds']),
                'Total Hours': round(part['Production_Hours'], 2),
                'Assignment Type': 'Flexible (balanced)'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Summary per machine (3-day view)
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Total Hours (3 days)'] = summary['Total Hours'].round(2)
    summary['Available Hours (3 days)'] = AVAILABLE_HOURS_PER_MACHINE
    summary['Utilization % (3 days)'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Load Status'] = summary['Utilization % (3 days)'].apply(
        lambda x: 'Overloaded (>100%)' if x > 100 else 
                  'High (85-100%)'     if x > 85 else 
                  'Good (60-85%)'      if x > 60 else 'Low / Underutilized'
    )
    
    summary = summary[['Part Count', 'Total Hours (3 days)', 'Available Hours (3 days)', 
                       'Utilization % (3 days)', 'Load Status']]
    
    return plan_df, summary


def main():
    print("=== 120 Ton Vertical Machines - 3 Day Production Plan ===\n")
    print(f"Capacity per machine: {HOURS_PER_DAY} hrs/day × {PLANNING_DAYS} days = {AVAILABLE_HOURS_PER_MACHINE} hrs\n")
    
    agg_df = load_and_aggregate_data()
    fixed, flexible = calculate_loads(agg_df)
    
    flexible_assignments, machine_load_sec = assign_flexible_parts(flexible)
    
    plan_df, summary = create_final_plan(fixed, flexible_assignments)
    
    # Save results
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_df.to_excel(writer, sheet_name='Detailed Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
    
    print("\n" + "="*75)
    print("              PRODUCTION PLAN GENERATED")
    print("="*75)
    
    print("\nMachine Summary (3-day plan, 22 productive hrs/day):")
    print(summary)
    
    print(f"\nDetailed assignment saved to: {OUTPUT_FILE}")
    
    overloaded = summary[summary['Utilization % (3 days)'] > 100]
    if not overloaded.empty:
        print("\nWARNING: Following machines are overloaded:")
        print(overloaded)
    else:
        print("\nNo overload detected. Plan is feasible within capacity.")
    
    # Optional: Daily average suggestion
    print("\nApproximate daily target per part (even split over 3 days):")
    daily_view = plan_df[['Machine', 'Child Part', 'Plan Qty (3 days)', 'Total Hours']].copy()
    daily_view['Daily Qty ≈'] = (daily_view['Plan Qty (3 days)'] / 3).round(0).astype(int)
    daily_view['Daily Hours ≈'] = (daily_view['Total Hours'] / 3).round(2)
    print(daily_view.sort_values(['Machine', 'Daily Hours ≈'], ascending=[True, False]))


if __name__ == "__main__":
    main()

=== 120 Ton Vertical Machines - 3 Day Production Plan ===

Capacity per machine: 22 hrs/day × 3 days = 66 hrs

Original rows in file     : 2232
Unique Child Parts        : 560
Parts duplicated (≥2x)    : 401
Fixed-assigned parts      : 0
Flexible parts to assign  : 560

              PRODUCTION PLAN GENERATED

Machine Summary (3-day plan, 22 productive hrs/day):
         Part Count  Total Hours (3 days)  Available Hours (3 days)  \
Machine                                                               
MP-01           127               4593.03                        66   
MP-05           177               4503.73                        66   
MP-10           128               4593.04                        66   
MP-17           128               4593.03                        66   

         Utilization % (3 days)         Load Status  
Machine                                              
MP-01                    6959.1  Overloaded (>100%)  
MP-05                    6823.8  Overloaded (>

In [10]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE AS NEEDED
# ────────────────────────────────────────────────
FILE_PATH = r"C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"   # ← CHANGE THIS to your actual file path
PLAN_SHEET = "Master Data "                                             # ← change if needed
OUTPUT_FILE = "production_plan_120ton.xlsx"                                  # ← your planning sheet name (change if different)
INVENTORY_SHEET = "Inventory "                            # as you confirmed



# 120 tonnage vertical machines
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Production capacity
HOURS_PER_DAY = 22
SECONDS_PER_DAY = HOURS_PER_DAY * 3600                   # 79,200 seconds/day
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS   # 237,600 seconds
AVAILABLE_HOURS_PER_MACHINE   = HOURS_PER_DAY * PLANNING_DAYS     # 66 hours

# Column names - Plan sheet
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_PLAN_QTY    = "Plan"
COL_CYCLE_TIME  = "Cycle Time"
COL_MACHINE     = "Vertical Machines"
COL_MIN_QTY     = "Minimum Quantity"

# Column names - Inventory sheet
COL_INVENTORY_MATERIAL = "Material"                      # child part reference
COL_INVENTORY_STOCK    = "Unrestricted"                  # current stock
# ────────────────────────────────────────────────

def load_plan_data():
    df_plan = pd.read_excel(FILE_PATH, sheet_name=PLAN_SHEET)
    df_plan.columns = df_plan.columns.str.strip()
    
    keep_cols = [COL_CHILD, COL_SWITCH, COL_PLAN_QTY, COL_CYCLE_TIME,
                 COL_MACHINE, COL_MIN_QTY]
    df_plan = df_plan[keep_cols].copy()
    
    numeric_cols = [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MIN_QTY]
    df_plan[numeric_cols] = df_plan[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    df_plan = df_plan.dropna(subset=[COL_CHILD, COL_PLAN_QTY, COL_CYCLE_TIME])
    
    # Take FIRST occurrence per Child Part
    df_plan = df_plan.sort_index()
    aggregated = df_plan.groupby(COL_CHILD, as_index=False).first()
    
    aggregated = aggregated.rename(columns={
        COL_PLAN_QTY:    'Plan_Qty',
        COL_CYCLE_TIME:  'Cycle_Time_s',
        COL_MACHINE:     'Assigned_Machine',
        COL_SWITCH:      'Switch_Reference',
        COL_MIN_QTY:     'Min_Qty'
    })
    
    print(f"Unique Child Parts from Plan sheet: {len(aggregated)}")
    return aggregated


def load_inventory_dict():
    df_inv = pd.read_excel(FILE_PATH, sheet_name=INVENTORY_SHEET)
    df_inv.columns = df_inv.columns.str.strip()
    
    # Keep only needed columns and clean
    df_inv = df_inv[[COL_INVENTORY_MATERIAL, COL_INVENTORY_STOCK]].copy()
    df_inv[COL_INVENTORY_STOCK] = pd.to_numeric(df_inv[COL_INVENTORY_STOCK], errors='coerce').fillna(0)
    
    # Create dict: Material -> Unrestricted stock
    inv_dict = dict(zip(df_inv[COL_INVENTORY_MATERIAL].astype(str).str.strip(),
                        df_inv[COL_INVENTORY_STOCK]))
    
    print(f"Inventory records loaded: {len(inv_dict)} unique materials")
    return inv_dict


def apply_inventory_logic(plan_df, inv_dict):
    plan_df['Current_Inventory'] = plan_df[COL_CHILD].astype(str).str.strip().map(inv_dict).fillna(0)
    
    # Production qty = max(0, max(Plan_Qty, Min_Qty) - Current_Inventory)
    plan_df['Produce_Qty'] = plan_df.apply(
        lambda row: max(0, max(row['Plan_Qty'], row['Min_Qty']) - row['Current_Inventory']),
        axis=1
    )
    
    # Only keep parts that need production
    needing_production = plan_df[plan_df['Produce_Qty'] > 0].copy()
    
    print(f"Parts needing production: {len(needing_production)} / {len(plan_df)} total unique")
    return needing_production


def calculate_loads(needing_df):
    needing_df['Production_Seconds'] = needing_df['Produce_Qty'] * needing_df['Cycle_Time_s']
    needing_df['Production_Hours']   = needing_df['Production_Seconds'] / 3600
    
    needing_df['Assigned_Machine'] = needing_df['Assigned_Machine'].astype(str).str.strip().str.upper()
    
    fixed = needing_df[needing_df['Assigned_Machine'].isin(ALLOWED_MACHINES)].copy()
    flexible = needing_df[~needing_df['Assigned_Machine'].isin(ALLOWED_MACHINES)].copy()
    
    print(f"Fixed parts needing production   : {len(fixed)}")
    print(f"Flexible parts needing production: {len(flexible)}")
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
    assignments = defaultdict(list)
    
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        best = min(machine_load, key=machine_load.get)
        assignments[best].append(row)
        machine_load[best] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    for _, row in fixed.iterrows():
        plan_rows.append({
            'Machine': row['Assigned_Machine'],
            'Child Part': row[COL_CHILD],
            'Switch Reference': row['Switch_Reference'],
            'Current Inventory': row['Current_Inventory'],
            'Min Qty': row['Min_Qty'],
            'Plan Qty (3 days)': row['Plan_Qty'],
            'Produce Qty': row['Produce_Qty'],
            'Cycle Time (s)': row['Cycle_Time_s'],
            'Total Seconds': int(row['Production_Seconds']),
            'Total Hours': round(row['Production_Hours'], 2),
            'Type': 'Fixed'
        })
    
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Reference': part['Switch_Reference'],
                'Current Inventory': part['Current_Inventory'],
                'Min Qty': part['Min_Qty'],
                'Plan Qty (3 days)': part['Plan_Qty'],
                'Produce Qty': part['Produce_Qty'],
                'Cycle Time (s)': part['Cycle_Time_s'],
                'Total Seconds': int(part['Production_Seconds']),
                'Total Hours': round(part['Production_Hours'], 2),
                'Type': 'Flexible (balanced)'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Summary
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Total Hours (3 days)'] = summary['Total Hours'].round(2)
    summary['Available Hours (3 days)'] = AVAILABLE_HOURS_PER_MACHINE
    summary['Utilization % (3 days)'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Load Status'] = summary['Utilization % (3 days)'].apply(
        lambda x: 'Overloaded (>100%)' if x > 100 else 
                  'High (85-100%)'     if x > 85 else 
                  'Good (60-85%)'      if x > 60 else 'Low'
    )
    
    summary = summary[['Part Count', 'Total Hours (3 days)', 'Available Hours (3 days)', 
                       'Utilization % (3 days)', 'Load Status']]
    
    return plan_df, summary


def main():
    print("=== 120 Ton - 3-Day Production Plan with Inventory Check ===\n")
    
    plan_df = load_plan_data()
    inv_dict = load_inventory_dict()
    
    needing_df = apply_inventory_logic(plan_df, inv_dict)
    if len(needing_df) == 0:
        print("No parts need production (all covered by inventory).")
        return
    
    fixed, flexible = calculate_loads(needing_df)
    flexible_assignments, _ = assign_flexible_parts(flexible)
    
    plan_output, summary = create_final_plan(fixed, flexible_assignments)
    
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_output.to_excel(writer, sheet_name='Production Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
        plan_df.to_excel(writer, sheet_name='All Unique Parts', index=False)  # for reference
    
    print("\n" + "="*80)
    print("         PRODUCTION PLAN GENERATED WITH INVENTORY CHECK")
    print("="*80)
    
    print("\nMachine Summary (only parts needing production):")
    print(summary)
    
    print(f"\nDetailed plan saved → {OUTPUT_FILE}")
    
    print("\nDaily approx targets (even split over 3 days - only for parts to produce):")
    daily = plan_output.copy()
    daily['Daily Produce ≈'] = (daily['Produce Qty'] / 3).round(0).astype(int)
    daily['Daily Hours ≈'] = (daily['Total Hours'] / 3).round(2)
    print(daily[['Machine', 'Child Part', 'Produce Qty', 'Daily Produce ≈', 'Daily Hours ≈']]
          .sort_values(['Machine', 'Daily Hours ≈'], ascending=[True, False]))


if __name__ == "__main__":
    main()

=== 120 Ton - 3-Day Production Plan with Inventory Check ===

Unique Child Parts from Plan sheet: 560
Inventory records loaded: 3104 unique materials
Parts needing production: 490 / 560 total unique
Fixed parts needing production   : 0
Flexible parts needing production: 490

         PRODUCTION PLAN GENERATED WITH INVENTORY CHECK

Machine Summary (only parts needing production):
         Part Count  Total Hours (3 days)  Available Hours (3 days)  \
Machine                                                               
MP-01           131               4557.70                        66   
MP-05           120               4557.65                        66   
MP-10           119               4557.64                        66   
MP-17           120               4557.62                        66   

         Utilization % (3 days)         Load Status  
Machine                                              
MP-01                    6905.5  Overloaded (>100%)  
MP-05                    6905

In [11]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx")

# Numeric cleanup
num_cols = [
    "Daily Plan", "Monthly Requirement", "Minimum Quantity",
    "Plan", "Inventory_25", "Cycle Time", "Main Count", "Sub Count"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# =============================
# Constants
# =============================
ALLOWED_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}
MACHINE_CAPACITY = 3960

# =============================
# Derived fields
# =============================
df["Net Required Qty"] = df["Plan"] - df["Inventory_25"]

df["Priority Score"] = (
    df["Main Count"] * 10
    + (df["Minimum Quantity"] - df["Inventory_25"]).clip(lower=0) * 5
    + df["Net Required Qty"].clip(lower=0) * 20
)

df = df.sort_values("Priority Score", ascending=False)

# =============================
# Mandatory load → target load
# =============================
mandatory_load = (
    df[df["Net Required Qty"] > 0]["Net Required Qty"]
    * df["Cycle Time"]
).sum()

TARGET_LOAD = mandatory_load / len(ALLOWED_MACHINES)

# =============================
# Tracking
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejections = []

# =============================
# Allocation
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    cycle = row["Cycle Time"]
    net_qty = row["Net Required Qty"]

    if cycle <= 0:
        continue

    vertical = [m.strip() for m in str(row["Vertical Machines"]).split(",")]
    eligible = [m for m in vertical if m in ALLOWED_MACHINES]

    if not eligible:
        if net_qty > 0:
            rejections.append((child, "No eligible machine"))
        continue

    # Decide allocatable qty
    if net_qty > 0:
        qty_to_make = net_qty
    else:
        qty_to_make = abs(net_qty)  # upper bound for balancing only

    remaining_time = qty_to_make * cycle

    def score(m, alloc):
        return abs((machine_load[m] + alloc) - TARGET_LOAD)

    eligible.sort(key=lambda m: score(
        m, min(MACHINE_CAPACITY - machine_load[m], remaining_time)
    ))

    for m in eligible:
        if remaining_time <= 0:
            break

        # Balancing-only restriction
        if net_qty <= 0 and machine_load[m] >= TARGET_LOAD:
            continue

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc = min(available, remaining_time)
        qty = alloc / cycle

        machine_load[m] += alloc
        remaining_time -= alloc

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(qty, 2),
            "Time (min)": round(alloc, 2)
        })

    if net_qty > 0 and remaining_time > 0:
        rejections.append((child, "Capacity shortfall"))

# =============================
# DISPLAY
# =============================
print("\n========== MACHINE PLAN ==========\n")
for m in sorted(ALLOWED_MACHINES):
    print(f"🔧 {m}")
    display(pd.DataFrame(machine_plan[m]))
    print("-" * 60)

print("\n========== MACHINE LOAD ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in sorted(ALLOWED_MACHINES)
]))

print("\n========== REJECTIONS ==========\n")
if rejections:
    display(pd.DataFrame(rejections, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")


========== MACHINE PLAN ==========

🔧 MP-01


""


------------------------------------------------------------
🔧 MP-05


""


------------------------------------------------------------
🔧 MP-10


...

""


------------------------------------------------------------

========== MACHINE LOAD ==========



,Machine,Used (min),Remaining (min)
0,MP-01,0,3960
1,MP-05,0,3960
2,MP-10,0,3960
3,MP-17,0,3960



========== REJECTIONS ==========



,Child Part,Reason
0,S22127-007A0X,No eligible machine
1,S22127-007A0X,No eligible machine
2,S22127-007A0X,No eligible machine
3,S22127-007A0X,No eligible machine
4,S22127-007A0X,No eligible machine
...,...,...
2035,S31583-004A1X,No eligible machine
2036,S31583-005A0X,No eligible machine
2037,S31583-001A1X,No eligible machine
2038,S31886-002A0X,No eligible machine


In [15]:
import pandas as pd
import numpy as np
from datetime import datetime

# Assumptions:
# - Excel file path: Replace 'your_file.xlsx' with your actual file path.
# - Sheet names: 'Main' for the primary data, 'inventory' for unrestricted stock, 'part_production_master' for cycle times.
# - Columns in 'Main' sheet (based on description):
#   - 'child_part': Unique ID for child parts.
#   - 'switch_part': Unique ID for switch parts.
#   - 'sub_count': Quantity of child part per switch part (int).
#   - 'inventory': Current stock (from main sheet, but we'll override with 'inventory' sheet).
#   - 'daily_plan_switch': Daily production target for each switch part (int).
#   - 'monthly_req_child': Monthly requirement for child parts (calculated or provided).
# - We calculate main_count as sum of sub_counts for each child across all switches.
# - Min quantity: (monthly_req / 31) * 3
# - Plan_child: min_qty + daily_req_child - (sub_count * 2)  # Per your example, but aggregated?
#   Note: Since sub_count is per switch-child pair, we'll assume the formula applies per pair and aggregate.
# - Machine details: changeover_time = 40 min, avail_hours_day = 22, efficiency = 0.85 (assumed).
# - Cycle times in 'part_production_master': Columns 'child_part', 'cycle_time_min' (minutes per unit).

# Script will:
# 1. Load data.
# 2. Calculate requirements and plans.
# 3. Check inventory and flag shortfalls.
# 4. Estimate machine capacity and suggest schedules.
# 5. Output to console and a new Excel file 'production_plan.xlsx'.

def main(file_path= "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"):
    # Load Excel
    xls = pd.ExcelFile(file_path)
    
    # Read sheets
    df_main = pd.read_excel(xls, sheet_name='Master Data ')  # Adjust sheet name if different
    df_inventory = pd.read_excel(xls, sheet_name='Inventory ')
    df_cycle = pd.read_excel(xls, sheet_name='Part Production Master ')
    
    # Merge unrestricted inventory into main (override main's inventory)
    df_main = df_main.merge(df_inventory[['Material', 'Unrestricted']], left_on='Child Part', right_on='Material', how='left')
    df_main['Inventory'] = df_main['Unrestricted'].fillna(df_main['Inventory'])  # Use unrestricted if available
    df_main.drop(['Material', 'Unrestricted'], axis=1, inplace=True)
    
    # Calculate daily_req_child per child-switch pair
    df_main['daily_req_pair'] = df_main['daily_plan_switch'] * df_main['Sub Count']
    
    # Aggregate for each child part
    df_child_agg = df_main.groupby('Child Part').agg({
        'daily_req_pair': 'sum',  # Total daily_req_child = sum over switches
        'Sub Count': 'sum',  # main_count = sum sub_counts
        'monthly_req_child': 'first',  # Assume provided per child, or calculate as sum(daily_req_pair * 25) for 25 working days
        'Inventory': 'first'
    }).reset_index()
    df_child_agg.rename(columns={'Sub Count': 'Main Count', 'daily_req_pair': 'daily_req_child'}, inplace=True)
    
    # If monthly_req not provided, estimate (adjust working_days as needed)
    working_days = 25  # Example: Exclude weekends/holidays
    if 'monthly_req_child' not in df_main.columns:
        df_child_agg['monthly_req_child'] = df_child_agg['daily_req_child'] * working_days
    
    # Min quantity
    df_child_agg['min_qty'] = (df_child_agg['monthly_req_child'] / 31) * 3
    
    # Plan_child: Adapted formula - since sub_count is now main_count, but per your example, use main_count * 2
    # In your clarification: For a single switch with sub_count=2, daily=600, req=1200
    # But formula is min_qty + daily_plan - (sub_count * 2), where daily_plan likely = daily_req
    df_child_agg['plan_child'] = df_child_agg['min_qty'] + df_child_agg['daily_req_child'] - (df_child_agg['main_count'] * 2)
    
    # Net needed: If negative, excess; positive = produce this much
    df_child_agg['net_needed'] = df_child_agg['plan_child'] - df_child_agg['inventory']
    df_child_agg['production_qty'] = np.maximum(df_child_agg['net_needed'], 0)
    
    # Merge cycle times
    df_child_agg = df_child_agg.merge(df_cycle[['child_part', 'cycle_time_min']], on='child_part', how='left')
    
    # Capacity calculations (assume 1 machine for simplicity; scale num_machines as needed)
    num_machines = 1
    changeover_min = 40
    avail_hours_day = 22
    efficiency = 0.85
    avail_min_day = avail_hours_day * 60 * efficiency * num_machines
    
    # Assume number of changeovers = number of child parts with production >0 minus 1 (if sequencing)
    num_producing = (df_child_agg['production_qty'] > 0).sum()
    changeovers = max(num_producing - 1, 0)  # Minimal if sequenced
    changeover_total_min = changeovers * changeover_min
    
    # Time needed per child
    df_child_agg['time_needed_min'] = df_child_agg['production_qty'] * df_child_agg['cycle_time_min']
    
    total_time_needed = df_child_agg['time_needed_min'].sum() + changeover_total_min
    load_percent = (total_time_needed / avail_min_day) * 100
    
    # Flags
    df_child_agg['shortfall_flag'] = df_child_agg['net_needed'] > 0
    df_child_agg['bottleneck_risk'] = (df_child_agg['main_count'] > df_child_agg['main_count'].median()) & df_child_agg['shortfall_flag']
    
    # Simple scheduling: Sort by priority (high main_count first)
    df_schedule = df_child_agg[df_child_agg['production_qty'] > 0].sort_values(by=['main_count', 'time_needed_min'], ascending=False)
    
    # Output to console
    print("Aggregated Child Part Data:")
    print(df_child_agg)
    print(f"\nTotal Time Needed: {total_time_needed:.2f} min")
    print(f"Available Capacity: {avail_min_day:.2f} min")
    print(f"Load: {load_percent:.2f}%")
    if load_percent > 100:
        print("WARNING: Overcapacity - Consider adding machines, overtime, or reducing plans.")
    
    print("\nSuggested Production Schedule:")
    print(df_schedule[['child_part', 'production_qty', 'time_needed_min', 'cycle_time_min']])
    
    # Export to new Excel
    with pd.ExcelWriter('production_plan.xlsx') as writer:
        df_child_agg.to_excel(writer, sheet_name='Child_Aggregates', index=False)
        df_schedule.to_excel(writer, sheet_name='Schedule', index=False)

if __name__ == "__main__":
    main()

KeyError: 'Inventory'